# Missing Value Handling

### 1. What Are Missing Values?

A missing value means that information expected for a particular observation is unavailable.

In Pandas, missing values are commonly represented by:

NaN

For example:

| customer_id | age | gender | monthly_charges |
| ----------- | --: | ------ | --------------: |
| CUST1001    |  25 | Female |            50.5 |
| CUST1002    | NaN | Male   |            65.2 |
| CUST1003    |  40 | NaN    |            72.1 |


Here:

* age is missing for customer 1002.
* gender is missing for customer 1003.

Real-world example:

A telecom customer may not have provided their age during registration.

#### AI/ML usage

Most ML algorithms cannot directly work with arbitrary missing values, so missing values usually need to be:

* removed,
* imputed,
* or handled by an algorithm that supports missing values.

### 2. Why Do Missing Values Occur?

Missing data can occur for many reasons.

Common reasons:

**1. Customer didn't provide information**

Example:

Age = missing

**2. Data collection failure**

A website or application may fail to record a value.

**3. Sensor/system failure**

A device may fail to capture a measurement.

**4. Not applicable**

Example:

* Internet service = No
* Data usage = Not applicable

**5. Data integration problem**

Two databases may have different fields.

**6. Human error**

An employee may forget to enter information.

**7. Privacy**

A customer may intentionally not disclose information.

### 3. Our Dataset – Detect Missing Values

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("telecom_customer_churn.csv")

df.head()

,customer_id,gender,age,tenure_months,monthly_charges,contract_type,internet_service,payment_method,support_calls,satisfaction_score,data_usage_gb,senior_citizen,paperless_billing,region,signup_date,total_charges,churn,customer_notes
0,CUST101731,female,36.0,20,68.69,One year,fiber optic,Credit card,4,9.0,7.5,No,Yes,Mumbai,2021-01-07,1528.44,No,Follow-up needed
1,CUST102657,Female,24.0,2,105.01,One year,No,credit card,2,1.0,40.6,Yes,Yes,Hyderabad,2021-04-12,214.51,No,VIP
2,CUST100034,Male,35.0,9,74.22,month-to-month,No,Electronic check,1,8.0,136.9,No,No,Delhi,2022-12-21,574.52,No,Requested callback
3,CUST104928,male,45.0,16,78.24,1 year,DSL,Mailed check,3,10.0,81.3,No,Yes,Hyderabad,2022-03-15,1157.27,No,Follow-up needed
4,CUST103902,NaN,36.0,43,66.66,Month-to-month,DSL,credit card,4,7.0,19.4,No,No,Delhi,2023-12-10,2601.54,No,Requested callback


In [3]:
# Check missing values:

df.isnull().sum()

customer_id             0
gender                127
age                    89
tenure_months           0
monthly_charges       114
contract_type          75
internet_service        0
payment_method        126
support_calls           0
satisfaction_score    102
data_usage_gb         152
senior_citizen          0
paperless_billing       0
region                 61
signup_date             0
total_charges         190
churn                   0
customer_notes          0
dtype: int64

### 4. Missing Value Percentage

Counting missing values alone isn't enough.

We should calculate the percentage.

Formula: 

$Missing Percentage = \frac{Total Rows}{Number of Missing Values} ×100$

In [5]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_percentage.sort_values(ascending=False)

total_charges         3.743842
data_usage_gb         2.995074
gender                2.502463
payment_method        2.482759
monthly_charges       2.246305
satisfaction_score    2.009852
age                   1.753695
contract_type         1.477833
region                1.201970
customer_id           0.000000
support_calls         0.000000
internet_service      0.000000
tenure_months         0.000000
senior_citizen        0.000000
paperless_billing     0.000000
signup_date           0.000000
churn                 0.000000
customer_notes        0.000000
dtype: float64

Why percentage matters

Suppose:

* Column A → 10 missing values out of 5,000
* Column B → 4,000 missing values out of 5,000

Both technically have missing values, but their impact is completely different.

###5. MCAR – Missing Completely At Random

MCAR means the probability of a value being missing is unrelated to both observed and unobserved data.

In simple words:

The value is missing for a completely random reason.

Example:

Suppose a server randomly fails to record the age of 100 customers.

The missingness has nothing to do with:

* age,
* gender,
* contract,
* charges,
* churn.

That is approximately MCAR.

Real-world example:

A database export randomly loses a few cells because of a technical issue.

#### ML usage

If data is genuinely MCAR and only a small percentage is missing, dropping affected rows may sometimes be reasonable.

In [6]:
df["age"].isnull().sum()

np.int64(89)

However, you generally cannot prove MCAR just by looking at isnull(). You need to investigate relationships between missingness and other variables.



### 6. MAR – Missing At Random

MAR means the probability of missingness depends on other observed variables.

Example:

Suppose older customers are less likely to provide their income.

Income:

* Age 25 → available
* Age 30 → available
* Age 65 → missing
* Age 70 → missing

The missingness may depend on age, which we can observe.

Telecom example:

Suppose customers with certain contract types are less likely to provide demographic information.

Then:

Missing gender -> related to -> contract_type

This could be MAR.

#### ML usage

MAR can often be handled using:

* group-based imputation,
* regression/iterative imputation,
* KNN,
* other model-based methods.

### 7. MNAR – Missing Not At Random

MNAR means the probability of missingness depends on the missing value itself or an unobserved factor.

Example:

Suppose customers with extremely high charges intentionally don't report their income.

So:

High income → more likely to hide income

The missingness itself is related to the value we don't observe.

Real-world example:

Customers with very low satisfaction may be more likely to skip a satisfaction survey.

Therefore:

Low satisfaction -> more likely to be missing

Important:

MNAR is difficult because the reason for missingness cannot usually be determined from the observed dataset alone.

We may need:

* domain knowledge,
* additional data,
* customer surveys,
* external information.

### 8. MCAR vs MAR vs MNAR

| Type | Meaning                             | Example                                       |
| ---- | ----------------------------------- | --------------------------------------------- |
| MCAR | Completely random                   | Random system failure                         |
| MAR  | Depends on observed data            | Income missing more often for older customers |
| MNAR | Depends on missing/unobserved value | Very dissatisfied customers avoid survey      |


### 9. Dropping Rows

Remove observations containing missing values.

In [7]:
df_drop_rows = df.dropna()

This removes any row containing at least one missing value.

Real-world example:

Suppose only 0.5% of customer records have missing information and you have millions of records.

Removing those records may be acceptable.

**When to use:**

* Missing percentage is very small.
* Dataset is large.
* Missing rows are not systematically different.
* The affected observations aren't especially important.

**When NOT to use:**

* Many rows contain missing values.
* Dataset is small.
* Missingness is systematic.
* Removing rows creates bias.

**Advantages:**

* Very simple.
* No artificial values are introduced.
* Easy to understand.

**Limitations:**

* Loses data.
* Can reduce sample size.
* Can introduce bias.
* Multiple columns with missing values can cause many rows to disappear.

In [8]:
print("Original:", len(df))
print("After drop:", len(df.dropna()))

Original: 5075
After drop: 4120


### 10. Dropping Columns

Remove an entire column when it has too many missing values or isn't useful.

In [9]:
df_drop_column = df.drop(
    columns=["customer_notes"]
)

customer_notes is also potentially irrelevant to our basic structured churn model.

**When to use:**

* A feature has an extremely high missing percentage.
* The feature isn't important.
* Reliable imputation isn't possible.

**Don't use when:**

* A column has moderate missingness and contains important predictive information.

**Advantages:**

* Simple.
* Reduces dimensionality.
* No need to impute that column.

**Limitations:**

* Can remove useful information.
* May reduce model performance.

### 11. Mean Imputation

Replace missing numerical values with the mean.

Example:

10

20

30

NaN

40

Mean:

$\frac{10+20+30+40}{4} =25$

Missing value becomes:

25

In [10]:
df["age"] = df["age"].fillna(
    df["age"].mean()
)

Real-world example:

If customer ages are approximately normally distributed and only a few ages are missing, mean imputation can be reasonable.


**When to use:**

* Numerical variable.
* Distribution is approximately symmetric.
* Few values are missing.
* Outliers aren't a major problem.

**When NOT to use:**

* Data is heavily skewed.
* Strong outliers exist.
* Missingness is systematic.

For example, total_charges can be highly skewed, so blindly using the mean isn't necessarily appropriate.

**Advantages:**

* Very easy.
* Fast.
* Keeps all rows.
  
**Limitations:**

* Reduces variability.
* Can distort the distribution.
* Sensitive to outliers.
* Can weaken relationships between variables.

### 12. Median Imputation

Replace missing numerical values with the median.

Example:

10

20

25

30

1000

NaN

Median = 25.

In [11]:
df["total_charges"] = df["total_charges"].fillna(
    df["total_charges"].median()
)

Why median can be better for our dataset:

Suppose monthly charges contain extreme values.

Mean: ₹250

Median: ₹75

The mean can be pulled upward by outliers.

Median is more robust.

**When to use:**

* Numerical variable.
* Data is skewed.
* Outliers exist.
* You want a simple robust method.

**When NOT to use:**

* The variable has a meaningful distribution that should be preserved.
* Missingness is strongly related to other features.

**Advantages:**

* Robust to outliers.
* Simple.
* Fast.

**Limitations:**

* Reduces variation.
* Doesn't use relationships with other variables.
* Can still distort the distribution.

**Good choices in our dataset**

Median can be considered for:

* age
* monthly_charges
* total_charges
* data_usage_gb
* satisfaction_score

after checking their distributions and business meaning.

### 13. Mode Imputation

The mode is the most frequently occurring value.

For categorical data:

Male
    
Female
    
Male
    
Male
    
Female
    
NaN

Mode = Male.

In [12]:
df["gender"] = df["gender"].fillna(
    df["gender"].mode()[0]
)

Real-world example:

If most customers are from Chennai and a small number of region values are missing, mode imputation may be considered.

**When to use:**

* Categorical variables.
* Low percentage of missing values.
* Stable categories.

**When NOT to use:**

* Categories are relatively balanced.
* Missingness is meaningful.
* The most frequent category would overwhelm the distribution.

**Advantages:**

* Simple.
* Works with categorical data.
* Easy to implement.

**Limitations:**

* Can increase the frequency of the most common category.
* Can distort category proportions.
* Doesn't use relationships with other features.

### 14. Constant Value Imputation

Replace missing values with a fixed value.

For example:

* Unknown
* Not Available
* 0

In [14]:
# Categorical example

df["region"] = df["region"].fillna(
    "Unknown"
)

In [15]:
# Numerical example

df["support_calls"] = df["support_calls"].fillna(0)

But only use 0 if zero has a meaningful business interpretation.

Real-world example:

If a customer didn't provide their region: region = Unknown is often more honest than pretending they belong to the most common region.

**When to use:**

* Missingness itself is meaningful.
* "Unknown" has a business interpretation.
* Zero genuinely means "none."

**Don't use when:**

Don't replace every missing numerical value with 0.

For example:

Age = 0 doesn't make sense for an adult telecom customer.

**Advantages:**

* Simple.
* Preserves rows.
* Makes missingness explicit.

**Limitations:**

* Can introduce artificial patterns.
* Zero can be misleading.
* Constant values can distort distributions.

### 15. Forward Fill – ffill

Forward fill replaces a missing value with the previous available value.

Example:

100

120

NaN

140

After forward fill:

100
    
120

120

140

In [17]:
df["monthly_charges"] = df["monthly_charges"].ffill()

Real-world example:

This is useful for time-series data.

For example, daily electricity readings:

Monday   ->  100
Tuesday  ->  105
Wednesday  ->  NaN
Thursday ->  110

Wednesday might be temporarily missing.

**When to use:**

* Time-series data.
* Sequential observations.
* Measurements where the previous value is meaningful.
  
**When NOT to use:**

* Random customer records.
* Age.
* Gender.
* Region.
* Customer contract type.

The previous customer's value doesn't tell us anything about the current customer.

**Advantages:**

* Simple.
* Maintains continuity.
* Useful for time-series.
  
**Limitations:**

* Can propagate incorrect values.
* Not appropriate for unrelated rows.
* Can create artificial patterns.

### 16. Backward Fill – bfill

Backward fill replaces a missing value with the next available value.

Example:

100

NaN

120

130

After:

100

120

120

130


In [18]:
df["monthly_charges"] = df["monthly_charges"].bfill()

Real-world example:

Useful in time-series data when the next observation is a reasonable estimate.

**When to use:**

* Time-series.
* Sequential data.
* Short gaps.

**When NOT to use:**

Don't use it on unrelated customer records.

For example:

* Customer A → Chennai
* Customer B → NaN
* Customer C → Mumbai

You cannot assume Customer B belongs to Mumbai.

**Advantages:**

* Simple.
* Useful for short time-series gaps.

**Limitations:**

* Can introduce incorrect values.
* Not suitable for general tabular customer data.

### 17. Interpolation

Interpolation estimates a missing value using surrounding values.

Example:

100

110

NaN

130

The missing value can be estimated as:

120

In [20]:
df["monthly_charges"] = pd.to_numeric(df["monthly_charges"], errors="coerce")

In [21]:
df["monthly_charges"] = df["monthly_charges"].interpolate()

Real-world example:

Temperature:

10 AM → 25°C
    
11 AM → NaN
    
12 PM → 29°C

Estimated 11 AM temperature might be around: 27°C

**When to use:**

* Time-series.
* Continuous measurements.
* Ordered numerical data.

**When NOT to use:**

* Gender.
* Region.
* Contract type.
* Random customer records without meaningful ordering.

**Advantages:**

* Uses surrounding values.
* Preserves trends.
* Better than constant filling for some continuous data.

**Limitations:**
    
* Assumes a meaningful relationship between neighboring observations.
* Not appropriate for categorical data.
* Can be inaccurate with sudden changes.

### 18. Group-Based Imputation

Instead of using one value for the entire dataset, calculate the replacement value separately for different groups.

For example:

Fill missing monthly charges using the median charge for that customer's contract type.

In [23]:
df["monthly_charges"] = df.groupby(
    "contract_type"
)["monthly_charges"].transform(
    lambda x: x.fillna(x.median())
)

Instead of:

All customers → one median

we get:

Month-to-month → its median
    
One year       → its median
    
Two year       → its median
    
Real-world example:

Monthly charges are likely to differ depending on contract type.

Therefore, contract-specific imputation may be more sensible.

**When to use:**

* Groups have meaningful differences.
* Another feature strongly relates to the missing variable.
    
**Don't use when:**
    
* Groups are very small.
* Group variable isn't related to the missing feature.
* Too many groups exist.
    
**Advantages:**
    
* More personalized.
* Uses business relationships.
* Often better than global mean/median.
    
**Limitations:**
    
* More complicated.
* Small groups can produce unreliable estimates.
* Requires choosing a meaningful grouping variable.

### 19. KNN Imputation 

KNN = K-Nearest Neighbors.

KNN imputation finds observations that are similar to the row with the missing value and uses their values to estimate the missing value.

Imagine:

Customer A
    
Age = 30
    
Tenure = 24

Charges = NaN

KNN searches for customers with similar:

* Age
* Tenure
* Support calls
* Data usage

and uses their charge values.


In [24]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=5)

numeric_columns = [
    "age",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "support_calls",
    "satisfaction_score",
    "data_usage_gb"
]

df[numeric_columns] = knn_imputer.fit_transform(
    df[numeric_columns]
)

Important

KNN uses distance.

Therefore, features with very different scales can affect the distance calculation.

In a proper ML workflow, scaling and imputation need to be designed carefully, usually inside a pipeline, to avoid leakage.

**When to use:**

* Numerical features are related.
* Dataset isn't enormous.
* Similar observations provide useful information.
    
**Don't use when:**
    
* Dataset is extremely large.
* Features aren't meaningfully related.
* Data contains many categorical variables without appropriate handling
    
**Advantages:**
    
* Uses relationships between observations.
* More intelligent than simple mean/median.
* Can preserve relationships better.
    
**Limitations:**
    
* Computationally expensive.
* Sensitive to scaling.
* Sensitive to choice of k.
*Can perform poorly if observations aren't meaningfully comparable.

### 20. Iterative Imputation

Iterative imputation predicts missing values using other features, repeatedly improving the estimates.

Instead of saying:

"Missing age = median age"

it essentially asks:

"Given the customer's other information, what age would be reasonable?"

Example:

To estimate missing monthly_charges, it may use:

* age
* tenure
* contract
* internet service
* support calls
* data usage

In [25]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(
    random_state=42
)

df[numeric_columns] = iterative_imputer.fit_transform(
    df[numeric_columns]
)

**When to use:**

* Several numerical features are related.
* Missingness is substantial.
* Relationships between variables can help predict missing values.

**Don't use when:**
    
* Dataset is extremely simple.
* Missing values are trivial.
* Computational simplicity is important.
* Relationships between features are weak.

**Advantages:**

* Uses relationships among variables.
* Usually more sophisticated than mean/median.
* Can preserve multivariate relationships better.

**Limitations:**
    
* More computationally expensive.
* More complicated.
* Estimates depend on the model.
* Can introduce model-based assumptions.